In [1]:
from surprise import accuracy, Dataset, SVD, Reader
from surprise.model_selection import train_test_split
import pandas as pd
from surprise.model_selection import cross_validate
from collections import defaultdict, Counter

df_ratings_total = pd.read_csv('ml-latest-small/ratings.csv').drop(columns='timestamp')
df_ratings_traing = pd.read_csv('data_spliting_cf1/training.csv').drop(columns='timestamp')
df_ratings_testing = pd.read_csv('data_spliting_cf1/testing.csv').drop(columns='timestamp')
df_movies = pd.read_csv('ml-latest-small/movies.csv')

reader = Reader(rating_scale=([0.5, 5]))

# Convertendo os dois separadamente
data_training = Dataset.load_from_df(df_ratings_traing, reader=reader)
data_testing = Dataset.load_from_df(df_ratings_testing, reader=reader)

'''
O Surprise espera dois formatos diferentes:

Trainset → um objeto especial interno do Surprise, usado pelo .fit()
Testset → uma lista de tuplas (uid, iid, nota_real), usado pelo .test()

'''

# Treino: build_full_trainset() — constrói um trainset com TODOS os dados do DataFrame
trainset = data_training.build_full_trainset()

# Teste: build_full_trainset().build_testset() — converte para lista de tuplas (uid, iid, nota_real)
testset = data_testing.build_full_trainset().build_testset()


algo = SVD(    
    n_factors=100,    # dimensões do espaço latente
    n_epochs=20,      # épocas de SGD
    lr_all=0.005,     # learning rate
    reg_all=0.02,     # regularização L2
    random_state=42,
    verbose=True
)

algo.fit(trainset)

#### Algoítimo aprendido:
print(f'vetor latente usuário: {algo.pu.shape}')
print(f'vetor latente filme: {algo.qi.shape}')
print(f'vetor de vies de cada usuário: {algo.bu.shape}')
print(f'vetor de vies de cada filme: {algo.bi.shape}')


## A ideia principal do modelo é prever nota
''' 
Aqui ele preve a nota do user 1 no filme 1
'''
print(algo.predict(uid=1, iid=1))

'''
Testar o modelo, comparando as predições feitas a partir do treinamento com os dados presentes no teste
exemplo predição do user 1 no filme 133:
Prediction(uid=1, iid=1030, r_ui=3.0, est=3.8121344471576815, details={'was_impossible': False}

'''
predicoes = algo.test(testset)


# NOTA: O SVD nunca marca was_impossible=True para filmes/usuários desconhecidos —
# ele usa a média global como fallback sem lançar PredictionImpossible.
# Por isso filtramos manualmente comparando com os IDs vistos no treino.
filmes_no_treino = set(df_ratings_traing['movieId'].unique())
usuarios_no_treino = set(df_ratings_traing['userId'].unique())



Processing epoch 0
Processing epoch 1
Processing epoch 2
Processing epoch 3
Processing epoch 4
Processing epoch 5
Processing epoch 6
Processing epoch 7
Processing epoch 8
Processing epoch 9
Processing epoch 10
Processing epoch 11
Processing epoch 12
Processing epoch 13
Processing epoch 14
Processing epoch 15
Processing epoch 16
Processing epoch 17
Processing epoch 18
Processing epoch 19
vetor latente usuário: (610, 100)
vetor latente filme: (8536, 100)
vetor de vies de cada usuário: (610,)
vetor de vies de cada filme: (8536,)
user: 1          item: 1          r_ui = None   est = 4.72   {'was_impossible': False}


In [12]:
def recomenda_usuario(usuario_id, top_n=10):
    filmes_assistidos = df_ratings_traing[df_ratings_traing['userId'] == usuario_id]['movieId'].tolist()

    filmes_nao_assistidos = sorted([
        movie_id for movie_id in filmes_no_treino
        if movie_id not in filmes_assistidos
    ])

    predicoes = []
    for id in filmes_nao_assistidos:
        pred = algo.predict(uid=usuario_id, iid=id)
        predicoes.append((id, pred.est))

    arr_sorted = sorted(predicoes, key=lambda x: x[1], reverse=True)

    recomendados = arr_sorted[:top_n]
    ids_recomendados = [movie_id for movie_id, nota in recomendados]

    df_resultado = df_movies.loc[df_movies['movieId'].isin(ids_recomendados), ['movieId', 'titulo_movie_lens']]

    return df_resultado


recomendados = recomenda_usuario(1)


In [13]:
def precisao_recall_usuario(usuario_id, top_n=10):
    ids_recomendados = recomenda_usuario(usuario_id, top_n)['movieId'].tolist()

    filmes_no_teste_usuario = df_ratings_testing[df_ratings_testing['userId'] == usuario_id]['movieId'].tolist()

    relevantes_recomendados = [m for m in ids_recomendados if m in filmes_no_teste_usuario]

    precisao = len(relevantes_recomendados) / len(ids_recomendados) if ids_recomendados else 0
    recall = len(relevantes_recomendados) / len(filmes_no_teste_usuario) if filmes_no_teste_usuario else 0

    print(f"Usuário: {usuario_id}")
    print(f"Filmes recomendados: {len(ids_recomendados)}")
    print(f"Filmes do usuário no teste: {len(filmes_no_teste_usuario)}")
    print(f"Relevantes recomendados: {len(relevantes_recomendados)}")
    print(f"Precisão: {precisao:.4f}")
    print(f"Recall:   {recall:.4f}")

    return precisao, recall


precisao_recall_usuario(1)


Usuário: 1
Filmes recomendados: 10
Filmes do usuário no teste: 70
Relevantes recomendados: 2
Precisão: 0.2000
Recall:   0.0286


(0.2, 0.02857142857142857)

In [ ]:
####Fazer alterações aqui

def precision_recall_at_k_v2(k=50, threshold=4.0):
    """
    Calcula Precision@K e Recall@K chamando recomenda_usuario para cada usuário.
    Um filme é considerado relevante se estiver no teste com rating >= threshold.
    """
    usuarios = df_ratings_testing['userId'].unique()

    precisions = {}
    recalls = {}

    for usuario_id in usuarios:
        ids_recomendados = recomenda_usuario(usuario_id, top_n=k)['movieId'].tolist()

        filmes_teste_usuario = df_ratings_testing[df_ratings_testing['userId'] == usuario_id]
        filmes_relevantes_teste = set(
            filmes_teste_usuario[filmes_teste_usuario['rating'] >= threshold]['movieId'].tolist()
        )

        n_rel_top_k = sum(1 for movie_id in ids_recomendados if movie_id in filmes_relevantes_teste)
        n_rel_total = len(filmes_relevantes_teste)

        precisions[usuario_id] = n_rel_top_k / k if k > 0 else 0
        recalls[usuario_id] = n_rel_top_k / n_rel_total if n_rel_total > 0 else 0

    mean_precision = sum(precisions.values()) / len(precisions) if precisions else 0
    mean_recall = sum(recalls.values()) / len(recalls) if recalls else 0

    return mean_precision, mean_recall, precisions, recalls


K = 10
mean_p, mean_r, precisions, recalls = precision_recall_at_k_v2(k=K, threshold=4.0)

print(f'Mean Precision@{K}: {mean_p:.4f}')
print(f'Mean Recall@{K}:    {mean_r:.4f}')
print(f'Usuários avaliados: {len(precisions)}')





Mean Precision@10: 0.0426
Mean Recall@10:    0.0248
Usuários avaliados: 610


In [15]:
def filmes_mais_recomendados(top_n_usuario=10, top_n_resultado=20):
    usuarios = df_ratings_traing['userId'].unique()
    todos_recomendados = []

    for usuario_id in usuarios:
        df_rec = recomenda_usuario(usuario_id, top_n_usuario)
        todos_recomendados.extend(df_rec['movieId'].tolist())

    contagem = Counter(todos_recomendados)
    mais_comuns = contagem.most_common(top_n_resultado)

    ids = [movie_id for movie_id, _ in mais_comuns]
    qtds = [qtd for _, qtd in mais_comuns]

    df_resultado = df_movies[df_movies['movieId'].isin(ids)][['movieId', 'titulo_movie_lens']].copy()
    df_resultado['vezes_recomendado'] = df_resultado['movieId'].map(dict(mais_comuns))
    df_resultado = df_resultado.sort_values('vezes_recomendado', ascending=False).reset_index(drop=True)

    return df_resultado


df_mais_recomendados = filmes_mais_recomendados(top_n_usuario=10, top_n_resultado=20)
df_mais_recomendados


,movieId,titulo_movie_lens,vezes_recomendado
0,750,Dr. Strangelove or: How I Learned to Stop Worr...,309
1,1204,Lawrence of Arabia (1962),259
2,318,"Shawshank Redemption, The (1994)",216
3,741,Ghost in the Shell (Kôkaku kidôtai) (1995),212
4,912,Casablanca (1942),169
5,1250,"Bridge on the River Kwai, The (1957)",128
6,2959,Fight Club (1999),119
7,1104,"Streetcar Named Desire, A (1951)",114
8,1225,Amadeus (1984),110
9,858,"Godfather, The (1972)",101
